In [0]:
# 1. Cargar librerias inicales
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

In [0]:
# 2. Crear la tabla workspace.weather_gold.dim_location
spark.sql("""
CREATE TABLE IF NOT EXISTS workspace.weather_gold.dim_location (
  department STRING,
  location_id STRING,
  latitude DOUBLE,
  longitude DOUBLE,
  timezone STRING,
  timezone_abbreviation STRING,
  elevation DOUBLE,
  weather_region_id INTEGER
)
""")

In [0]:

# 2. Leer bronze y pasar a pandas
df_silver = spark.table("workspace.weather_silver.weather")
df_silver.show(5, truncate=False)

In [0]:

# 3. Crear DIM_LOCATION
df_silver=df_silver.toPandas()
dim_location = (
    df_silver[
        [
            "department",
            "location_id",
            "latitude",
            "longitude",
            "timezone",
            "timezone_abbreviation",
            "elevation"
        ]
    ]
    .drop_duplicates()
)

# 4. Agregar identificador de region(Costa, Sierra, Selva) esto para la dimension region
# 4. Agregar identificador de región
dim_location["weather_region_id"] = (
    dim_location["department"].map(
        {
            "Tumbes": 1,
            "Piura": 1,
            "Lambayeque": 1,
            "La_Libertad": 1,
            "Ancash": 1,
            "Lima": 1,
            "Ica": 1,
            "Arequipa": 1,
            "Moquegua": 1,
            "Tacna": 1,
            "Callao": 1,

            "Cajamarca": 2,
            "Huanuco": 2,
            "Pasco": 2,
            "Junin": 2,
            "Huancavelica": 2,
            "Ayacucho": 2,
            "Apurimac": 2,
            "Cusco": 2,
            "Puno": 2,

            "Amazonas": 3,
            "Loreto": 3,
            "San_Martin": 3,
            "Ucayali": 3,
            "Madre_de_Dios": 3
        }
    )
)
# 5. Ordenar por departamento
dim_location = dim_location.sort_values(
    "department"
)

# 6. Ver resultado
print(dim_location.head())
print(dim_location.shape)

# 7. Pandas -> Spark
dim_location_spark = spark.createDataFrame(dim_location)
dim_location_spark.printSchema()

# 8. Guardar la dimension DIM_LOCATION
dim_location_spark.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("workspace.weather_gold.dim_location")

In [0]:
%sql
select * from workspace.weather_gold.dim_location limit 50